In [7]:
# Use %pip to ensure installation into the CURRENT notebook kernel
%pip install "numpy<2" opencv-python joblib scikit-image scikit-learn tensorflow keras

  Obtaining dependency information for numpy<2 from https://files.pythonhosted.org/packages/16/2e/86f24451c2d530c88daf997cb8d6ac622c1d40d19f5a031ed68a4b73a374/numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------ ------------------------- 20.5/61.0 kB 222.6 kB/s eta 0:00:01
     ------------ ------------------------- 20.5/61.0 kB 222.6 kB/s eta 0:00:01
     ------------------- ------------------ 30.7/61.0 kB 131.3 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/61.0 kB 179.6 kB/s eta 0:00:01
     ------------------------------- ------ 51.2/61.0 kB 187.9 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 191.7 kB/s eta 0:00:00
  Obtaining dependency information for opencv-python from https://files.pythonhosted.org/packages/e9/a5/1be1516390333ff9be3a9cb648c9f33df79d5096e5884b5df71a588af463/ope

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

print("OpenCV Version:", cv2.__version__)
print("NumPy Version:", np.__version__)

OpenCV Version: 4.11.0
NumPy Version: 1.26.4


In [9]:
IMG_SIZE = 128

def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    features = hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    return features

In [10]:
import os
import cv2
import numpy as np
from skimage.feature import hog

# -----------------------------
# 1. CONFIG
# -----------------------------
IMG_SIZE = 128

# Auto-detect if running in Colab or locally
if os.path.exists('/content'):
    BASE_PATH = "/content"
else:
    # Search for dataset in current or parent directories
    BASE_PATH = os.getcwd()
    # Try to find 'alzheimer' directory up to 3 levels up
    for _ in range(3):
        if os.path.isdir(os.path.join(BASE_PATH, 'alzheimer')):
            break
        if os.path.isdir(os.path.join(BASE_PATH, 'backend', 'alzheimer')):
            BASE_PATH = os.path.join(BASE_PATH, 'backend')
            break
        BASE_PATH = os.path.dirname(BASE_PATH)

AUGMENTED_PATH = None
ORIGINAL_PATH = None

# -----------------------------
# 2. AUTO DETECT DATASET PATHS
# -----------------------------
search_dirs = [BASE_PATH, os.path.join(BASE_PATH, 'alzheimer'), os.path.join(BASE_PATH, 'backend', 'alzheimer')]

for d in search_dirs:
    if not os.path.isdir(d): continue
    
    orig = os.path.join(d, "OriginalDataset")
    if os.path.isdir(orig):
        ORIGINAL_PATH = orig
    
    aug_names = ["AugmentedAlzheimerDataset", "AugmentedDataset", "Augmented Alzheimer Dataset"]
    for name in aug_names:
        aug = os.path.join(d, name)
        if os.path.isdir(aug):
            AUGMENTED_PATH = aug
            break

print("Augmented Path:", AUGMENTED_PATH)
print("Original Path:", ORIGINAL_PATH)

if not AUGMENTED_PATH and not ORIGINAL_PATH:
    print("WARNING: Dataset not found. Current Search Path:", BASE_PATH)

Augmented Path: d:\CODE\NeuroSense\backend\alzheimer\AugmentedAlzheimerDataset
Original Path: d:\CODE\NeuroSense\backend\alzheimer\OriginalDataset


In [ ]:
import time
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# 4. LOAD DATASET
# -----------------------------
data = []
labels = []

dataset_path = AUGMENTED_PATH if AUGMENTED_PATH else ORIGINAL_PATH

if not dataset_path:
    raise ValueError("Dataset path not found. Please check your folder structure.")

classes = [c for c in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, c))]
print("Classes found:", classes)

for label in classes:
    folder = os.path.join(dataset_path, label)
    files = [f for f in os.listdir(folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"Loading {label} (Found {len(files)} images)...")
    
    for file in files:
        path = os.path.join(folder, file)
        try:
            features = extract_features(path)
            data.append(features)
            labels.append(label)
        except Exception as e:
            pass

data = np.array(data)
labels = np.array(labels)

if len(data) == 0:
    raise ValueError("No data loaded. Check image formats and paths.")

print(f"\nDataset loaded: {len(data)} samples.")

start_total = time.time()

# -----------------------------
# 1. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

svm = SVC(kernel='rbf', probability=True)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100)

ensemble = VotingClassifier(
    estimators=[('svm', svm), ('rf', rf), ('gb', gb)],
    voting='soft'
)

print("Training Ensemble model...")
ensemble.fit(X_train, y_train)

print("Evaluating...")
y_pred = ensemble.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

joblib.dump(ensemble, "alzheimers_ml_ensemble.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Model saved as alzheimers_ml_ensemble.pkl")

Classes found: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
Loading MildDemented (Found 8960 images)...
Loading ModerateDemented (Found 6464 images)...
Loading NonDemented (Found 9600 images)...
Loading VeryMildDemented (Found 8960 images)...

Dataset loaded: 33984 samples.
Training Ensemble model...


In [4]:
import os
import cv2
import time
import joblib
import numpy as np

from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# CONFIG
# -----------------------------
# FIXED: Points to the 'alzheimer' subdirectory where your data lives
ORIGINAL_PATH = os.path.join("alzheimer", "OriginalDataset")
AUGMENTED_PATH = os.path.join("alzheimer", "AugmentedAlzheimerDataset")
IMG_SIZE = 128

CLASSES = [
    "NonDemented",
    "VeryMildDemented",
    "MildDemented",
    "ModerateDemented"
]

# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_features(image):
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )
    return features

# -----------------------------
# LOAD DATASET
# -----------------------------
def load_dataset():
    data = []
    labels = []

    for label, class_name in enumerate(CLASSES):
        print(f"Loading {class_name}...")

        # Scan both original and augmented folders
        count = 0
        for base_path in [ORIGINAL_PATH, AUGMENTED_PATH]:
            class_path = os.path.join(base_path, class_name)

            if not os.path.exists(class_path):
                continue

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                image = cv2.imread(img_path)
                if image is None:
                    continue

                features = extract_features(image)
                data.append(features)
                labels.append(label)
                count += 1
        print(f"  Done. Total images for {class_name}: {count}")

    return np.array(data), np.array(labels)

# -----------------------------
# MAIN
# -----------------------------
if __name__ == "__main__":

    start_total = time.time()

    print("Loading dataset (this may take a few minutes)...")
    data, labels = load_dataset()

    print("Data shape:", data.shape)

    # FIXED: Added check to prevent crash if data is empty
    if len(data) == 0:
        print("\nFATAL ERROR: No images were loaded.")
        print(f"Checked paths: \n - {os.path.abspath(ORIGINAL_PATH)}\n - {os.path.abspath(AUGMENTED_PATH)}")
        print("Please verify these folders contain the class subfolders.")
    else:
        print("Splitting dataset...")
        X_train, X_test, y_train, y_test = train_test_split(
            data, labels, test_size=0.2, random_state=42, stratify=labels
        )

        print("Scaling...")
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        print("Training FAST Logistic Regression...")
        start_train = time.time()

        model = LogisticRegression(
            max_iter=500,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        print("Training finished in", 
              round((time.time() - start_train)/60, 2), 
              "minutes")

        print("\nEvaluating...")
        y_pred = model.predict(X_test)
        print("Accuracy:", accuracy_score(y_test, y_pred))
        print("\nClassification Report:\n")
        print(classification_report(y_test, y_pred))

        print("Saving model...")
        joblib.dump(model, "alzheimers_fastest.pkl")
        joblib.dump(scaler, "scaler.pkl")

        print("Done. Model saved as 'alzheimers_fastest.pkl'")
        print("Total execution time:", 
              round((time.time() - start_total)/60, 2), 
              "minutes")


Loading dataset (this may take a few minutes)...
Loading NonDemented...
  Done. Total images for NonDemented: 12800
Loading VeryMildDemented...
  Done. Total images for VeryMildDemented: 11200
Loading MildDemented...
  Done. Total images for MildDemented: 9856
Loading ModerateDemented...
  Done. Total images for ModerateDemented: 6528
Data shape: (40384, 8100)
Splitting dataset...
Scaling...
Training FAST Logistic Regression...


c:\Users\tanis\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\tanis\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training finished in 7.62 minutes

Evaluating...
Accuracy: 0.783706821839792

Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.77      0.77      2560
           1       0.71      0.72      0.72      2240
           2       0.77      0.78      0.77      1971
           3       0.94      0.93      0.93      1306

    accuracy                           0.78      8077
   macro avg       0.80      0.80      0.80      8077
weighted avg       0.78      0.78      0.78      8077

Saving model...
Done. Model saved as 'alzheimers_fastest.pkl'
Total execution time: 22.37 minutes
